In [0]:
%run ../utils

In [0]:
process_table('student_sentiment', 'student_sentiment_processed')

In [0]:
display(spark.table('crm_ingestion.silver.student_sentiment_processed'))

In [0]:
# Exploration: full raw-column lineage with datatypes (scalars included).
# raw_lineage_tree() with no args scans all 9 bronze tables.
raw_lineage_tree('student_sentiment', 'student_sentiment_processed')

In [0]:
# TEMP — verify spec null-handling: any Python None/True/False leaked as STRING literals into silver?
from pyspark.sql import functions as F

tables = [
    'leads_processed', 'lead_activites_processed', 'custom_activites_processed',
    'calendly_scheduled_events_processed', 'student_sentiment_processed',
    'mdl_users_processed', 'close_crm_users_processed',
    'leads_processed_custom', 'leads_processed_contacts', 'leads_processed_contacts_emails'
]
for t in tables:
    try:
        df = spark.table(f'crm_ingestion.silver.{t}')
        str_cols = [f.name for f in df.schema.fields if f.dataType.simpleString() == 'string']
        if not str_cols:
            print(f'{t}: (no string cols)')
            continue
        cond_none = F.or_(*[F.col(c) == 'None' for c in str_cols])
        cond_bool = F.or_(*[F.col(c).isin('True', 'False') for c in str_cols])
        n_none = df.filter(cond_none).count()
        n_bool = df.filter(cond_bool).count()
        print(f'{t}: None-strings={n_none:,}  True/False-strings={n_bool:,}')
    except Exception as e:
        print(f'{t}: !! {str(e)[:80]}')

In [0]:
# Exploration: complex-table demo — arrays/structs recursed, scalars + datatypes
# printed, arrays show the silver child table they would become.
raw_lineage_tree('lead_activites_raw', 'lead_activites_processed')